# Step3: IT 섹터 Walk-Forward 분류 (XGBoost + TabPFN)

**설계 원칙**: 각 walk-forward 윈도우에서 IT 10 종목 각각에 대해 **독립적인** ML 모델 학습  
- XGBoost (Optuna 30 trials) + TabPFN v2 병렬 비교
- Target: 영업일 기준 21일 후 단순 수익률 5분위 분류
- Output: `Test/output/Q_xgb.csv`, `Omega_xgb.csv`, `Q_tabpfn.csv`, `Omega_tabpfn.csv`


In [1]:
"""
Section 0: Import 및 함수 정의
  - 모든 함수는 이 셀에서 정의 (노트북 독립 실행 가능)
"""
import io
import os
import sys
import platform
import warnings
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# XGBoost
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# TabPFN v0.1.9 (사전학습 분류기, 252행 이내 적합)
from tabpfn import TabPFNClassifier

# sklearn
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import LabelEncoder

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Windows 한글 출력 인코딩
if sys.stdout.encoding and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

warnings.filterwarnings("ignore")

# ─── 한글 폰트 설정 ───────────────────────────────────────────────────────────
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    try:
        import koreanize_matplotlib  # noqa: F401
    except ImportError:
        pass
plt.rcParams["axes.unicode_minus"] = False

# ─── 경로 설정 ────────────────────────────────────────────────────────────────

def _find_base_dir() -> Path:
    # data/panels 디렉터리가 있는 black_litterman 루트를 탐색한다.
    cwd = Path(os.getcwd()).resolve()
    for candidate in [cwd, cwd.parent, cwd.parent.parent]:
        if (candidate / "data" / "panels").is_dir():
            return candidate
    # Fallback: 절대 경로
    return Path("C:/Users/gorhk/최종 프로젝트/finance_project/김재천/black_litterman")

BASE_DIR     = _find_base_dir()
NOTEBOOK_DIR = BASE_DIR / "Test"
DATA_DIR     = BASE_DIR / "data"
OUT_DIR      = NOTEBOOK_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── 파라미터 ─────────────────────────────────────────────────────────────────
IT_TICKERS    = ["MSFT", "INTC", "ORCL", "AAPL", "CSCO",
                 "IBM", "QCOM", "TXN", "CRM", "ADBE"]
FEATURE_COLS  = [
    "log_return_1d", "simple_return_1d",
    "mom_1m", "mom_3m", "mom_6m", "mom_12m", "mom_12m_skip_1m",
    "vol_20d_ann", "vol_60d_ann", "vol_252d_ann",
    "mkt_rf", "smb", "hml", "rmw", "cma", "rf", "mom_factor",
]
TARGET_COL    = "fwd_ret_21d"
DATA_START    = pd.Timestamp("2020-12-01")
DATA_END      = pd.Timestamp("2025-12-31")
IS_DAYS       = 252
EMBARGO_DAYS  = 21
OOS_DAYS      = 21
STEP_SIZE     = 21
N_CLASSES     = 5
N_OPTUNA_TRIALS = 30
RANDOM_STATE  = 42


# ─── 함수 정의 ────────────────────────────────────────────────────────────────

def load_ticker_csv(ticker: str) -> pd.DataFrame:
    """
    panels/{ticker}.csv 로드 → DATA_START~DATA_END 필터 → fwd_ret_21d 계산.
    반환: date index, feature 컬럼 + fwd_ret_21d 포함 DataFrame
    """
    path = DATA_DIR / "panels" / f"{ticker}.csv"
    df = pd.read_csv(path, index_col="date", parse_dates=True)
    df = df[(df.index >= DATA_START) & (df.index <= DATA_END)].copy()
    df.sort_index(inplace=True)

    # 21일 후 단순 수익률 — 마지막 21행은 NaN
    df[TARGET_COL] = df["adj_close"].shift(-OOS_DAYS) / df["adj_close"] - 1
    return df


def make_wf_windows(
    dates: pd.DatetimeIndex,
    is_days: int = IS_DAYS,
    embargo: int = EMBARGO_DAYS,
    oos_days: int = OOS_DAYS,
    step: int = STEP_SIZE,
) -> List[Tuple[pd.DatetimeIndex, pd.DatetimeIndex]]:
    """
    Walk-forward 윈도우 생성.
    각 원소: (is_dates, oos_dates)
    IS 마지막 oos_days 행은 purge 처리 (fwd_ret 누출 방지).
    """
    windows = []
    n = len(dates)
    i = 0
    while True:
        is_end    = i + is_days           # IS 끝 인덱스 (exclusive)
        oos_start = is_end + embargo      # embargo 간격 후 OOS 시작
        oos_end   = oos_start + oos_days
        if oos_end > n:
            break
        # IS purge: 마지막 oos_days 행 제거 (forward-label 누출 방지)
        purge_cutoff = is_end - oos_days
        is_dates_w  = dates[i : purge_cutoff]
        oos_dates_w = dates[oos_start : oos_end]
        if len(is_dates_w) > 0 and len(oos_dates_w) > 0:
            windows.append((is_dates_w, oos_dates_w))
        i += step
    return windows


def make_quintile_labels(
    y_is: pd.Series,
) -> Tuple[np.ndarray, np.ndarray, Dict[int, float]]:
    """
    IS 수익률을 5분위로 나눠 레이블 생성.
    IS bins, IS 분위별 평균 수익률(r_bar) 반환.
    레이블: 0(최하위) ~ 4(최상위)
    """
    _, bins = pd.qcut(y_is.dropna(), q=N_CLASSES, retbins=True, duplicates="drop")
    bins[0]  = -np.inf
    bins[-1] =  np.inf

    labels = pd.cut(y_is, bins=bins, labels=False).fillna(0).astype(int)
    r_bar  = {k: y_is[labels == k].mean() for k in range(N_CLASSES)}

    return labels.values, bins, r_bar


def apply_bins(y: pd.Series, bins: np.ndarray) -> np.ndarray:
    """IS에서 계산된 bins를 OOS 데이터에 적용."""
    labels = pd.cut(y, bins=bins, labels=False).fillna(0).astype(int)
    return labels.values


def compute_Q_Omega(
    proba: np.ndarray,
    r_bar: Dict[int, float],
    n_classes: int = N_CLASSES,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    각 행(예측 시점)에 대해 B-L view Q와 불확실성 Ω 계산.

    Q_i   = Σ_k P(class=k) * r_bar[k]          # 가중 평균 기대 수익률
    Ω_i   = Σ_k P(class=k) * (r_bar[k] - Q_i)² # 분산 = 예측 불확실성
    """
    r_arr = np.array([r_bar.get(k, 0.0) for k in range(n_classes)])
    Q     = proba @ r_arr                                               # (n,)
    Omega = np.sum(proba * (r_arr[np.newaxis, :] - Q[:, np.newaxis]) ** 2, axis=1)  # (n,)
    return Q, Omega


def _xgb_objective(trial, X_tr, y_tr, X_val, y_val):
    """Optuna objective: XGBoost 하이퍼파라미터 최적화 (log_loss 최소화)."""
    params = {
        "n_estimators"      : trial.suggest_int("n_estimators",   100, 500),
        "max_depth"         : trial.suggest_int("max_depth",       3,   7),
        "learning_rate"     : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"         : trial.suggest_float("subsample",     0.5, 1.0),
        "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight"  : trial.suggest_int("min_child_weight", 1, 10),
        "gamma"             : trial.suggest_float("gamma",         0.0, 3.0),
        "reg_lambda"        : trial.suggest_float("reg_lambda",    0.1, 10.0, log=True),
        "reg_alpha"         : trial.suggest_float("reg_alpha",     1e-5, 1.0, log=True),
        "objective"         : "multi:softprob",
        "num_class"         : N_CLASSES,
        "eval_metric"       : "mlogloss",
        "tree_method"       : "hist",
        "random_state"      : RANDOM_STATE,
        "n_jobs"            : -1,
        "verbosity"         : 0,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    proba_val = model.predict_proba(X_val)
    # labels 명시 — 검증 분할에 일부 분위가 없을 때 클래스 수 불일치 방지
    return log_loss(y_val, proba_val, labels=list(range(N_CLASSES)))


def train_xgb(
    X_is: np.ndarray,
    y_is: np.ndarray,
    X_oos: np.ndarray,
    window_id: int = 0,
    n_trials: int = N_OPTUNA_TRIALS,
) -> Tuple[np.ndarray, dict]:
    """
    IS 데이터로 Optuna XGBoost 학습 → OOS 확률 반환.
    내부 80/20 분할로 하이퍼파라미터 탐색 후 전체 IS 재학습.
    """
    n_is  = len(X_is)
    split = int(n_is * 0.8)
    X_tr, X_val = X_is[:split], X_is[split:]
    y_tr, y_val = y_is[:split], y_is[split:]

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE + window_id)
    study   = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(
        lambda trial: _xgb_objective(trial, X_tr, y_tr, X_val, y_val),
        n_trials=n_trials,
        show_progress_bar=False,
    )

    best_params = study.best_params
    best_params.update({
        "objective"    : "multi:softprob",
        "num_class"    : N_CLASSES,
        "eval_metric"  : "mlogloss",
        "tree_method"  : "hist",
        "random_state" : RANDOM_STATE,
        "n_jobs"       : -1,
        "verbosity"    : 0,
    })

    # 전체 IS 재학습
    final_model = xgb.XGBClassifier(**best_params)
    final_model.fit(X_is, y_is, verbose=False)

    proba = final_model.predict_proba(X_oos)  # (n_oos, 5)
    return proba, best_params


def train_tabpfn(
    X_is: np.ndarray,
    y_is: np.ndarray,
    X_oos: np.ndarray,
) -> np.ndarray:
    """
    TabPFN v0.1.9 학습 → OOS 확률 반환.
    사전학습 모델이므로 Optuna 불필요.
    N_ensemble_configurations: 앙상블 구성 수 (v0.1.9 API 파라미터명)
    overwrite_warning=True: IS 행 수 경고 억제
    """
    clf = TabPFNClassifier(device="cpu", N_ensemble_configurations=32)
    clf.fit(X_is, y_is, overwrite_warning=True)
    proba = clf.predict_proba(X_oos)  # (n_oos, 5)

    # 클래스 순서가 0~4 순서임을 보장
    classes = clf.classes_
    if not np.array_equal(classes, np.arange(N_CLASSES)):
        order = np.argsort(classes)
        proba = proba[:, order]
    return proba


def compute_eval_metrics(
    Q_pred: np.ndarray,
    y_pred_cls: np.ndarray,
    y_true_cls: np.ndarray,
    y_true_ret: np.ndarray,
) -> Dict[str, float]:
    """
    분류 성능 지표 계산.
    - Accuracy : 5분위 분류 정확도
    - Hit Rate : P(sign(Q) == sign(actual_return))
    - IC       : Spearman(Q, actual_return)
    (LogLoss 는 window 단위로 별도 집계)
    """
    valid = ~np.isnan(y_true_ret)
    if valid.sum() == 0:
        return {"accuracy": np.nan, "hit_rate": np.nan, "ic": np.nan}

    acc      = accuracy_score(y_true_cls[valid], y_pred_cls[valid])
    hit_rate = np.mean(np.sign(Q_pred[valid]) == np.sign(y_true_ret[valid]))
    ic, _    = spearmanr(Q_pred[valid], y_true_ret[valid])

    return {"accuracy": acc, "hit_rate": hit_rate, "ic": ic if not np.isnan(ic) else 0.0}


print("Section 0 PASS [OK] — 함수 정의 완료")
print(f"  IT_TICKERS : {IT_TICKERS}")
print(f"  FEATURE_COLS ({len(FEATURE_COLS)}) : {FEATURE_COLS}")
print(f"  IS_DAYS={IS_DAYS}, EMBARGO={EMBARGO_DAYS}, OOS={OOS_DAYS}")

Section 0 PASS [OK] — 함수 정의 완료
  IT_TICKERS : ['MSFT', 'INTC', 'ORCL', 'AAPL', 'CSCO', 'IBM', 'QCOM', 'TXN', 'CRM', 'ADBE']
  FEATURE_COLS (17) : ['log_return_1d', 'simple_return_1d', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'mom_12m_skip_1m', 'vol_20d_ann', 'vol_60d_ann', 'vol_252d_ann', 'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'mom_factor']
  IS_DAYS=252, EMBARGO=21, OOS=21


## Section 1: 데이터 로드 및 설정 확인

In [2]:
"""
Section 1: 데이터 로드 테스트 — IT 10 종목 CSV 존재 확인, 날짜·피처 검증
"""
print("=" * 60)
print("SECTION 1: 데이터 로드 및 설정 확인")
print("=" * 60)

# 각 종목 로드 확인
all_dates = None
for ticker in IT_TICKERS:
    df = load_ticker_csv(ticker)

    # 피처 컬럼 존재 여부 확인
    missing_feats = [c for c in FEATURE_COLS if c not in df.columns]
    assert not missing_feats, f"{ticker} 피처 누락: {missing_feats}"

    # target 컬럼 확인
    assert TARGET_COL in df.columns, f"{ticker} target 컬럼 없음"

    n_total  = len(df)
    n_valid  = df[TARGET_COL].notna().sum()
    date_rng = f"{df.index[0].date()} ~ {df.index[-1].date()}"
    print(f"  {ticker:5s} : {n_total}행, target 유효 {n_valid}행, 기간 {date_rng}")

    if all_dates is None:
        all_dates = df.index

# Walk-forward 윈도우 생성 테스트
windows = make_wf_windows(all_dates)
print(f"\n  총 walk-forward 윈도우: {len(windows)}개")
print(f"  첫 번째 IS: {windows[0][0][0].date()} ~ {windows[0][0][-1].date()}")
print(f"  첫 번째 OOS: {windows[0][1][0].date()} ~ {windows[0][1][-1].date()}")
print(f"  마지막 OOS: {windows[-1][1][0].date()} ~ {windows[-1][1][-1].date()}")

print("\nSECTION 1 PASS [OK]")


SECTION 1: 데이터 로드 및 설정 확인
  MSFT  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  INTC  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  ORCL  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  AAPL  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  CSCO  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  IBM   : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  QCOM  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  TXN   : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  CRM   : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  ADBE  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30

  총 walk-forward 윈도우: 47개
  첫 번째 IS: 2020-12-01 ~ 2021-10-29
  첫 번째 OOS: 2021-12-31 ~ 2022-01-31
  마지막 OOS: 2025-11-06 ~ 2025-12-05

SECTION 1 PASS [OK]


## Section 2: Walk-Forward 실행 (XGBoost + TabPFN)

In [3]:
"""
Section 2: Walk-Forward 메인 루프

루프 구조:
  for window in windows:
    for ticker in IT_TICKERS:   ← 종목별 독립 반복
        XGBoost 학습 + 예측
        TabPFN 학습 + 예측
        Q, Ω 계산
    record OOS date별 Q, Ω
"""
print("=" * 60)
print("SECTION 2: Walk-Forward 실행")
print("=" * 60)

# 결과 누적 리스트
# 형식: [{'date': ..., 'ticker': ..., 'model': ...,
#          'Q': ..., 'Omega': ..., 'actual_ret': ...,
#          'pred_cls': ..., 'true_cls': ..., 'logloss': ...}]
records = []

# 종목별 데이터 사전 로드 (반복 IO 방지)
ticker_data = {t: load_ticker_csv(t) for t in IT_TICKERS}
all_dates   = list(ticker_data.values())[0].index
windows     = make_wf_windows(all_dates)

t0 = time.time()

for w_idx, (is_dates, oos_dates) in enumerate(windows):
    is_start  = is_dates[0].date()
    oos_start = oos_dates[0].date()
    print(f"\n[Window {w_idx+1:3d}/{len(windows)}] IS {is_start} ~ {is_dates[-1].date()}"
          f" | OOS {oos_start} ~ {oos_dates[-1].date()}")

    for ticker in IT_TICKERS:
        df_t = ticker_data[ticker]

        # ── IS / OOS 데이터 추출 ───────────────────────────────────────
        df_is  = df_t.loc[is_dates].dropna(subset=FEATURE_COLS + [TARGET_COL])
        df_oos = df_t.loc[df_t.index.isin(oos_dates)]

        if len(df_is) < 50 or len(df_oos) == 0:
            # 유효 데이터 부족 → 스킵
            print(f"  {ticker}: IS {len(df_is)}행 부족, 스킵")
            continue

        X_is   = df_is[FEATURE_COLS].values.astype(float)
        y_is   = df_is[TARGET_COL]
        X_oos  = df_oos[FEATURE_COLS].fillna(0).values.astype(float)
        y_oos  = df_oos[TARGET_COL].values  # 일부 NaN 가능 (마지막 OOS)

        # ── IS 5분위 레이블 생성 ───────────────────────────────────────
        try:
            y_is_cls, bins, r_bar = make_quintile_labels(y_is)
        except ValueError:
            # 분위 생성 실패 (중복 경계 등) → 스킵
            print(f"  {ticker}: 분위 생성 실패, 스킵")
            continue

        y_oos_cls = apply_bins(df_oos[TARGET_COL], bins)

        # ── XGBoost ────────────────────────────────────────────────────
        try:
            proba_xgb, _ = train_xgb(X_is, y_is_cls, X_oos, window_id=w_idx)
            Q_xgb, Om_xgb = compute_Q_Omega(proba_xgb, r_bar)
            ll_xgb = log_loss(y_oos_cls, proba_xgb, labels=list(range(N_CLASSES)))
            pred_cls_xgb = np.argmax(proba_xgb, axis=1)
        except Exception as e:
            print(f"  {ticker} XGB 오류: {e}")
            Q_xgb = np.zeros(len(X_oos))
            Om_xgb = np.ones(len(X_oos)) * 1e-4
            ll_xgb = np.nan
            pred_cls_xgb = np.zeros(len(X_oos), dtype=int)

        # ── TabPFN v2 ──────────────────────────────────────────────────
        try:
            proba_tab = train_tabpfn(X_is, y_is_cls, X_oos)
            Q_tab, Om_tab = compute_Q_Omega(proba_tab, r_bar)
            ll_tab = log_loss(y_oos_cls, proba_tab, labels=list(range(N_CLASSES)))
            pred_cls_tab = np.argmax(proba_tab, axis=1)
        except Exception as e:
            print(f"  {ticker} TabPFN 오류: {e}")
            Q_tab = np.zeros(len(X_oos))
            Om_tab = np.ones(len(X_oos)) * 1e-4
            ll_tab = np.nan
            pred_cls_tab = np.zeros(len(X_oos), dtype=int)

        # ── OOS 날짜별 결과 기록 ───────────────────────────────────────
        for j, oos_date in enumerate(df_oos.index):
            base = {
                "window_id"  : w_idx,
                "date"       : oos_date,
                "ticker"     : ticker,
                "actual_ret" : y_oos[j] if j < len(y_oos) else np.nan,
                "true_cls"   : y_oos_cls[j] if j < len(y_oos_cls) else -1,
            }
            if j < len(Q_xgb):
                records.append({**base, "model": "xgb",
                                 "Q": Q_xgb[j], "Omega": Om_xgb[j],
                                 "pred_cls": pred_cls_xgb[j], "logloss": ll_xgb})
            if j < len(Q_tab):
                records.append({**base, "model": "tabpfn",
                                 "Q": Q_tab[j], "Omega": Om_tab[j],
                                 "pred_cls": pred_cls_tab[j], "logloss": ll_tab})

        elapsed = time.time() - t0
        print(f"  {ticker}: XGB ll={ll_xgb:.4f} | TabPFN ll={ll_tab:.4f} ({elapsed:.0f}s 경과)")

elapsed_total = time.time() - t0
print(f"\n총 소요 시간: {elapsed_total/60:.1f}분")
print(f"총 기록 수: {len(records)}개")
print("SECTION 2 PASS [OK]")


SECTION 2: Walk-Forward 실행

[Window   1/47] IS 2020-12-01 ~ 2021-10-29 | OOS 2021-12-31 ~ 2022-01-31
Loading model that can be used for inference only
Using a Transformer with 25.82 M parameters


c:\Users\gorhk\최종 프로젝트\finance_project\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  MSFT: XGB ll=2.1815 | TabPFN ll=4.9661 (9s 경과)
Loading model that can be used for inference only
Using a Transformer with 25.82 M parameters


c:\Users\gorhk\최종 프로젝트\finance_project\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  INTC: XGB ll=1.3076 | TabPFN ll=0.5748 (21s 경과)
Loading model that can be used for inference only
Using a Transformer with 25.82 M parameters


...

  TXN: XGB ll=1.9897 | TabPFN ll=2.6961 (4172s 경과)
Loading model that can be used for inference only
Using a Transformer with 25.82 M parameters


c:\Users\gorhk\최종 프로젝트\finance_project\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  CRM: XGB ll=1.7862 | TabPFN ll=2.6372 (4180s 경과)
Loading model that can be used for inference only
Using a Transformer with 25.82 M parameters


c:\Users\gorhk\최종 프로젝트\finance_project\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  ADBE: XGB ll=2.1464 | TabPFN ll=4.3962 (4191s 경과)

총 소요 시간: 69.8분
총 기록 수: 19740개
SECTION 2 PASS [OK]


## Section 3: 분류 성능 평가

In [4]:
"""
Section 3: 분류 성능 평가
  - Accuracy, Hit Rate, IC, LogLoss
  - 종목별 × 모델별 요약표
  - 모델 전체 평균 비교 (XGBoost vs TabPFN)
"""
print("=" * 60)
print("SECTION 3: 분류 성능 평가")
print("=" * 60)

df_res = pd.DataFrame(records)

# 유효 데이터만 사용 (actual_ret NaN 제외)
df_valid = df_res.dropna(subset=["actual_ret"])

# 종목 × 모델별 집계
eval_rows = []
for (ticker, model), grp in df_valid.groupby(["ticker", "model"]):
    q_pred   = grp["Q"].values
    cls_pred = grp["pred_cls"].values
    cls_true = grp["true_cls"].values
    ret_true = grp["actual_ret"].values

    metrics = compute_eval_metrics(q_pred, cls_pred, cls_true, ret_true)
    eval_rows.append({
        "ticker"   : ticker,
        "model"    : model,
        "accuracy" : metrics["accuracy"],
        "hit_rate" : metrics["hit_rate"],
        "ic"       : metrics["ic"],
        "logloss"  : grp["logloss"].mean(),
        "n_oos"    : len(grp),
    })

df_eval = pd.DataFrame(eval_rows)

# 종목 × 모델 성능표
print("\n[종목 × 모델 성능표]")
print(df_eval.to_string(index=False, float_format="{:.4f}".format))

# 모델 전체 평균 비교
print("\n[모델 전체 평균 비교]")
agg = df_eval.groupby("model")[["accuracy", "hit_rate", "ic", "logloss"]].mean()
agg["accuracy_vs_random"]  = agg["accuracy"]  - 0.20
agg["hit_rate_vs_random"]  = agg["hit_rate"]  - 0.50
print(agg.to_string(float_format="{:.4f}".format))
print("  (accuracy 기준선=0.20, hit_rate 기준선=0.50, IC 기준선=0.0)")

print("\nSECTION 3 PASS [OK]")


SECTION 3: 분류 성능 평가

[종목 × 모델 성능표]
ticker  model  accuracy  hit_rate      ic  logloss  n_oos
  AAPL tabpfn    0.2067    0.4644 -0.0704   2.5313    982
  AAPL    xgb    0.1894    0.4674 -0.2466   1.7802    982
  ADBE tabpfn    0.1365    0.4766 -0.1071   3.1315    982
  ADBE    xgb    0.1558    0.4654 -0.0863   2.0538    982
   CRM tabpfn    0.1976    0.4695 -0.0553   2.7852    982
   CRM    xgb    0.1222    0.4409 -0.2241   1.9312    982
  CSCO tabpfn    0.1436    0.3992 -0.2042   3.2301    982
  CSCO    xgb    0.1079    0.4287 -0.1321   2.0580    982
   IBM tabpfn    0.3116    0.6100  0.3316   2.4496    982
   IBM    xgb    0.2403    0.6161  0.2527   1.7380    982
  INTC tabpfn    0.1568    0.4654 -0.1582   2.7036    982
  INTC    xgb    0.1904    0.4674 -0.1839   1.9238    982
  MSFT tabpfn    0.1853    0.4246 -0.2676   2.9463    982
  MSFT    xgb    0.1548    0.4379 -0.1585   1.9543    982
  ORCL tabpfn    0.1925    0.4695 -0.0826   2.6680    982
  ORCL    xgb    0.1527    0.4674 -0.

## Section 4: 결과 저장 (Step4 입력 파일)

In [5]:
"""
Section 4: Q, Ω CSV 저장 (Step4 B-L 입력용)

저장 파일:
  Test/output/Q_xgb.csv       — date × ticker, 연환산 21일 예측 수익률
  Test/output/Omega_xgb.csv   — date × ticker, 연환산 예측 불확실성
  Test/output/Q_tabpfn.csv
  Test/output/Omega_tabpfn.csv
  Test/output/classification_stats.csv
"""
print("=" * 60)
print("SECTION 4: 결과 저장")
print("=" * 60)

ANN_FACTOR  = 252 / OOS_DAYS          # 연환산 수익률 배수 = 12
ANN_FACTOR2 = ANN_FACTOR ** 2         # 연환산 분산 배수 = 144

for model_name in ["xgb", "tabpfn"]:
    df_m = df_res[df_res["model"] == model_name][["date", "ticker", "Q", "Omega"]].copy()

    # 연환산 변환
    df_m["Q"]     *= ANN_FACTOR
    df_m["Omega"] *= ANN_FACTOR2

    # date × ticker 피벗
    q_pivot     = df_m.pivot_table(index="date", columns="ticker", values="Q")
    omega_pivot = df_m.pivot_table(index="date", columns="ticker", values="Omega")

    # 컬럼 순서 통일 (IT_TICKERS 순서)
    q_pivot     = q_pivot.reindex(columns=IT_TICKERS)
    omega_pivot = omega_pivot.reindex(columns=IT_TICKERS)

    q_path     = OUT_DIR / f"Q_{model_name}.csv"
    omega_path = OUT_DIR / f"Omega_{model_name}.csv"
    q_pivot.to_csv(q_path,     encoding="utf-8-sig")
    omega_pivot.to_csv(omega_path, encoding="utf-8-sig")

    print(f"  [{model_name}] Q shape={q_pivot.shape}, Omega shape={omega_pivot.shape}")
    print(f"    Q 범위: [{q_pivot.values[~np.isnan(q_pivot.values)].min():.4f},"
          f" {q_pivot.values[~np.isnan(q_pivot.values)].max():.4f}]")
    print(f"    저장: {q_path.name}, {omega_path.name}")

# 분류 성능 저장
stats_path = OUT_DIR / "classification_stats.csv"
df_eval.to_csv(stats_path, index=False, encoding="utf-8-sig")
print(f"\n  classification_stats.csv 저장 완료 ({len(df_eval)}행)")

print("\nSECTION 4 PASS [OK]")
print("\n" + "=" * 60)
print("Step3 전체 완료 — Step4_IT_BlackLitterman.ipynb 실행 가능")
print("=" * 60)


SECTION 4: 결과 저장
  [xgb] Q shape=(987, 10), Omega shape=(987, 10)
    Q 범위: [-2.6324, 2.4134]
    저장: Q_xgb.csv, Omega_xgb.csv
  [tabpfn] Q shape=(987, 10), Omega shape=(987, 10)
    Q 범위: [-3.1017, 3.0686]
    저장: Q_tabpfn.csv, Omega_tabpfn.csv

  classification_stats.csv 저장 완료 (20행)

SECTION 4 PASS [OK]

Step3 전체 완료 — Step4_IT_BlackLitterman.ipynb 실행 가능
